# Decode the two public Starlink VHF events



In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

def find_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'project_config.py').is_file():
            return candidate
    raise FileNotFoundError('Run from this repository or one of its subdirectories')

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from project_config import DATA_DIR, RAW_DATA_DIR, RESULTS_DIR, file_start_utc
from project_config import verify_raw_data

verify_raw_data(calculate_md5=False)
OUTPUT_DIR = RESULTS_DIR / '2_demodulation'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if shutil.which('lfdemod') is None:
    raise RuntimeError('lfdemod is not installed; create the environment from environment.yml')


In [ ]:
jobs = [
    ('file0', RAW_DATA_DIR / '20250415-2200-0.dat', 5.0, 1.5),
    ('file12', RAW_DATA_DIR / '20250415-2200-12.dat', 2.6, 1.7),
]
for label, input_path, start_s, duration_s in jobs:
    output_path = OUTPUT_DIR / label
    command = [
        'lfdemod', 'starlink-vhf', '-i', str(input_path),
        '-f', '137.055e6', '-s', str(start_s), '-d', str(duration_s),
        '-o', str(output_path),
    ]
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True)
    if not (output_path / 'summary.json').is_file():
        raise FileNotFoundError(output_path / 'summary.json')
print('Decoded products:', OUTPUT_DIR)
